# WEPAStacks: MLP → ACO → Drone

Notebook này là giao diện chạy pipeline MLP–ACO/PSO mới. Logic chính được giữ trong các module Python để có thể unit test và tái sử dụng. Notebook MLP cũ không bị thay đổi.

Dữ liệu thật: thời điểm và inbound point của pallet events. Giả định mô phỏng: bốn kho vệ tinh, capacity, tọa độ cục bộ, depot và đội drone.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'src':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config_loader import load_config, resolve_project_path
from src.data_pipeline import load_interval_table
from src.metrics import operational_metrics
from src.predict_risk import RiskPredictor
from src.rolling_simulation import run_simulation
from src.train_mlp_aco import prepare_training_data, train

config = load_config(PROJECT_ROOT / 'config' / 'experiment.yaml')
config

## 1. Dữ liệu 15 phút

Mỗi time bin phải có đủ bốn warehouse rows; interval không có arrival được điền bằng 0.

In [ ]:
intervals = load_interval_table(config)
print('Rows:', len(intervals))
print('Time bins:', intervals['time_bin'].nunique())
print('Warehouses per bin:', intervals.groupby('time_bin')['dock'].nunique().unique())
display(intervals.head(8))

## 2. Historical queue và target

Historical state được tạo bằng fixed `fullest_first` policy. Target tại thời điểm `t` kiểm tra congestion trong 60 phút tiếp theo nếu không có drone; future arrivals không xuất hiện trong feature.

In [ ]:
model_table, split_masks = prepare_training_data(config)
for name, mask in zip(['train', 'validation', 'test'], split_masks):
    labels = model_table.loc[mask, 'congestion']
    print(name, 'samples =', len(labels), 'positive rate =', f'{labels.mean():.2%}')
display(model_table.head())

## 3. Huấn luyện MLP

Đổi `RUN_TRAINING` thành `True` khi muốn train lại. Model, scaler, threshold và metadata sẽ được lưu trong `model_outputs_aco/`.

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    training_metrics = train(config)
    display(training_metrics)
else:
    print('Training skipped. Set RUN_TRAINING=True to create model_outputs_aco/.')

## 4. Rolling MLP–ACO smoke experiment

Cell này chỉ chạy khi model mới tồn tại. Để thử nhanh, nó dùng 20 intervals, 5 ants và 10 iterations. Thí nghiệm chính thức vẫn dùng cấu hình đầy đủ và 10 seeds qua `run_experiment.py`.

In [ ]:
from copy import deepcopy

model_directory = resolve_project_path(config['outputs']['model_directory'], config)
model_path = model_directory / 'mlp_congestion_aco.keras'
if not model_path.exists():
    print('Chưa có model mới. Hãy chạy section 3 hoặc: python run_training.py')
else:
    smoke_config = deepcopy(config)
    smoke_config['aco']['ants'] = 5
    smoke_config['aco']['iterations'] = 10
    smoke_config['aco']['stall_iterations'] = 4
    test_start = int(config['mlp']['validation_end_day']) + 1
    test_intervals = intervals.loc[intervals['day'].ge(test_start)].copy()
    selected_bins = sorted(test_intervals['time_bin'].unique())[:20]
    test_intervals = test_intervals.loc[test_intervals['time_bin'].isin(selected_bins)]
    predictor = RiskPredictor(model_directory)
    interval_log, route_log = run_simulation(
        test_intervals, smoke_config, 'mlp_aco', predictor=predictor, seed=1
    )
    display(operational_metrics(interval_log))
    display(interval_log.head(8))
    display(route_log.head(6))

## 5. Thí nghiệm đầy đủ

Chạy ngoài notebook để lưu log riêng cho nearest-first, fullest-first, ACO-current, MLP-ACO, PSO-current và MLP-PSO:

```bash
python run_experiment.py
```

Kết quả được lưu trong `outputs/logs`, `outputs/routes` và `outputs/tables`.